# 7번 통신 처리량 저하 예측

- 타깃: +1~+15초 실측 dl_kbps 평균 < 20,000 → 1, 아니면 0
- 지표: macro-F1 (`labels=[0,1], average="macro", zero_division=0`)
- 계획: PLAN.md v2. 후보 A(규칙) → B(로지스틱 회귀, 임계값 0.5) → C(B + 중첩 임계값 선택). 검증은 세션 단위 LOSO

In [1]:
import time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
warnings.filterwarnings('ignore')
T0 = time.time()
R = 'data/'
tr = pd.read_csv(R+'train.csv', encoding='utf-8-sig'); te = pd.read_csv(R+'test.csv', encoding='utf-8-sig')
se = pd.read_csv(R+'sessions.csv', encoding='utf-8-sig'); lg = pd.read_csv(R+'link_log.csv', encoding='utf-8-sig')
ss = pd.read_csv(R+'sample_submission.csv', encoding='utf-8-sig')
for c in ['rsrq_db','snr_db','cqi','rssi_dbm','nr_rsrp_dbm','nr_rsrq_db']:   # '-' 결측 → NaN (EDA 재현용, 이번 피처에서는 미사용)
    lg[c] = pd.to_numeric(lg[c], errors='coerce')
lg['ldl'] = np.log1p(lg.dl_kbps); lg['low'] = (lg.dl_kbps < 20000).astype(float)
print(tr.shape, te.shape, lg.shape, ss.shape)

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


(387, 4) (168, 2) (21445, 16) (168, 2)


## 1. 샘플별 이력 피처 (5초 = -4..0, 15초 = -14..0, 45초 = 전체)

In [2]:
def feats(g):
    g = g.sort_values('relative_second')
    r, d = g.relative_second.values, g.ldl.values
    m5, m15 = r >= -4, r >= -14
    return pd.Series({
        'dl_raw45': g.dl_kbps.mean(),
        'ldl_m5': d[m5].mean(), 'ldl_m15': d[m15].mean(), 'ldl_m45': d.mean(),
        'low_f45': g.low.mean(),
        'ldl_min15': d[m15].min(), 'ldl_sd15': d[m15].std(),
        'slope15': np.polyfit(r[m15], d[m15], 1)[0],
        'n5': m5.sum(), 'n15': m15.sum(),
    })
X = lg.groupby('sample_id').apply(feats)
FEATS = ['ldl_m5','ldl_m15','ldl_m45','low_f45','ldl_min15','ldl_sd15','slope15']
tr = tr.merge(se, on='session_id').join(X, on='sample_id')
te = te.merge(se, on='session_id').join(X, on='sample_id')
assert tr[FEATS].notna().all().all() and te[FEATS].notna().all().all()
print('min rows: 5s', int(X.n5.min()), '/ 15s', int(X.n15.min()))
y = tr.label.values; g = tr.session_id.values; st = (tr.mobility == 'stationary').values
pt = (te.mobility == 'stationary').mean()
w = np.where(st, pt/st.mean(), (1-pt)/(1-st.mean()))   # test mobility 비중 보정 가중치
print(f'stationary share train {st.mean():.3f} / test {pt:.3f}; weights stat {w[st][0]:.3f}, mov {w[~st][0]:.3f}')

min rows: 5s 3 / 15s 11
stationary share train 0.279 / test 0.536; weights stat 1.920, mov 0.644


## 2. 안전본: 후보 A (과거 45초 dl 평균 < 20k) 제출 저장

In [3]:
def save_sub(pred_te, note):
    sub = ss[['sample_id']].merge(te[['sample_id']].assign(label=pred_te.astype(int)), on='sample_id', how='left')
    assert len(sub) == len(ss) and (sub.sample_id.values == ss.sample_id.values).all()
    assert sub.label.notna().all() and set(sub.label.unique()) <= {0, 1}
    sub.to_csv('submission.csv', index=False)
    print(f'saved submission.csv ({note}): n={len(sub)}, positive rate={sub.label.mean():.3f}')
predA_te = (te.dl_raw45 < 20000).astype(int).values
save_sub(predA_te, 'A safe')
print(f'elapsed {time.time()-T0:.1f}s')

saved submission.csv (A safe): n=168, positive rate=0.375
elapsed 0.4s


## 3. LOSO 비교 (A, B, C)

In [4]:
LOGO = list(LeaveOneGroupOut().split(tr, y, g))
TGRID = np.round(np.arange(0.30, 0.701, 0.05), 2)
LR = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
mf1 = lambda yt, p, sw=None: f1_score(yt, p, labels=[0,1], average='macro', zero_division=0, sample_weight=sw)

def pick_t(yt, p):   # macro-F1 최대, 동점이면 0.5에 가까운 값
    return max(TGRID, key=lambda t: (round(mf1(yt, (p > t).astype(int)), 4), -abs(t - 0.5)))

def loso_proba(Xdf, yy, splits):
    p = np.zeros(len(yy))
    for a, b in splits:
        p[b] = LR().fit(Xdf.iloc[a][FEATS], yy[a]).predict_proba(Xdf.iloc[b][FEATS])[:, 1]
    return p

predA = (tr.dl_raw45 < 20000).astype(int).values
oofB = loso_proba(tr, y, LOGO); predB = (oofB > 0.5).astype(int)

predC = np.zeros(len(y), int); chosen = {}
for a, b in LOGO:
    sub = tr.iloc[a].reset_index(drop=True)
    inner = list(LeaveOneGroupOut().split(sub, y[a], g[a]))
    t = pick_t(y[a], loso_proba(sub, y[a], inner))
    chosen[g[b][0]] = t
    predC[b] = LR().fit(tr.iloc[a][FEATS], y[a]).predict_proba(tr.iloc[b][FEATS])[:, 1] > t

sess_ids = pd.unique(g)
sess = lambda p: np.array([mf1(y[g == k], p[g == k]) for k in sess_ids])
sA = sess(predA)
rows = []
for name, p in [('A rule45<20k', predA), ('B LR@0.5', predB), ('C LR nested-thr', predC)]:
    s = sess(p)
    rows.append({'cand': name, 'all': mf1(y, p), 'test_mix_w': mf1(y, p, w), 'moving': mf1(y[~st], p[~st]),
                 'stationary': mf1(y[st], p[st]), 'win_vs_A': int((s > sA + 1e-9).sum()), 'loss_vs_A': int((s < sA - 1e-9).sum())})
res = pd.DataFrame(rows).set_index('cand').round(4)
print(res)
print('C chosen threshold per held-out session:', chosen)
print('reference only (same-OOF pick, optimistic):', {t: round(mf1(y, (oofB > t).astype(int)), 4) for t in TGRID})
print(f'elapsed {time.time()-T0:.1f}s')

                    all  test_mix_w  moving  stationary  win_vs_A  loss_vs_A
cand                                                                        
A rule45<20k     0.8262      0.8032  0.8280      0.6453         0          0
B LR@0.5         0.8324      0.8062  0.8459      0.6936         8          4
C LR nested-thr  0.8339      0.8070  0.8470      0.6791         9          2
C chosen threshold per held-out session: {'L001': 0.65, 'L002': 0.6, 'L003': 0.65, 'L004': 0.65, 'L005': 0.65, 'L006': 0.6, 'L007': 0.6, 'L008': 0.55, 'L009': 0.65, 'L010': 0.55, 'L011': 0.65, 'L017': 0.55, 'L018': 0.65, 'L019': 0.6}
reference only (same-OOF pick, optimistic): {0.3: 0.8088, 0.35: 0.8157, 0.4: 0.8198, 0.45: 0.8207, 0.5: 0.8324, 0.55: 0.841, 0.6: 0.8364, 0.65: 0.8445, 0.7: 0.8293}
elapsed 1.5s


## 4. 사전 채택 규칙 (PLAN §5) 적용

In [5]:
def passes(new, prev):
    r = res.loc
    checks = {
        'all >= prev': r[new, 'all'] >= r[prev, 'all'],
        'test_mix_w >= prev': r[new, 'test_mix_w'] >= r[prev, 'test_mix_w'],
        'wins >= losses (vs A)': r[new, 'win_vs_A'] >= r[new, 'loss_vs_A'],
        'stationary >= A': r[new, 'stationary'] >= r['A rule45<20k', 'stationary'],
    }
    print(f'{new} vs {prev}:', checks)
    return all(checks.values())

adopted = 'A rule45<20k'
for cand in ['B LR@0.5', 'C LR nested-thr']:
    if passes(cand, adopted):
        adopted = cand
    else:
        break
print('ADOPTED:', adopted)

B LR@0.5 vs A rule45<20k: {'all >= prev': True, 'test_mix_w >= prev': True, 'wins >= losses (vs A)': True, 'stationary >= A': True}
C LR nested-thr vs B LR@0.5: {'all >= prev': True, 'test_mix_w >= prev': True, 'wins >= losses (vs A)': True, 'stationary >= A': True}
ADOPTED: C LR nested-thr


## 5. train 전체 재학습 → test 예측 → 제출

In [6]:
if adopted == 'A rule45<20k':
    final_t = None; pred_te = predA_te
else:
    final_t = 0.5 if adopted.startswith('B') else pick_t(y, oofB)   # C: 전체 train LOSO OOF로 같은 규칙 적용
    model = LR().fit(tr[FEATS], y)
    p_te = model.predict_proba(te[FEATS])[:, 1]
    pred_te = (p_te > final_t).astype(int)
    print('final threshold:', final_t)
    print('coef:', dict(zip(FEATS, model[-1].coef_[0].round(3))))
save_sub(pred_te, adopted)
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == ['sample_id', 'label'] and len(chk) == 168 and chk.sample_id.equals(ss.sample_id)
print('A vs final agreement on test:', round((pred_te == predA_te).mean(), 3))
print(te.assign(pred=pred_te).groupby('mobility').pred.agg(['mean', 'size']))
print(f'elapsed {time.time()-T0:.1f}s')

final threshold: 0.65
coef: {'ldl_m5': -0.188, 'ldl_m15': -1.185, 'ldl_m45': 0.508, 'low_f45': 1.264, 'ldl_min15': -0.815, 'ldl_sd15': -1.322, 'slope15': -0.237}
saved submission.csv (C LR nested-thr): n=168, positive rate=0.363
A vs final agreement on test: 0.952
                mean  size
mobility                  
moving      0.692308    78
stationary  0.077778    90
elapsed 1.5s


## 6. (참고) 제공 fit/valid 분할 평가 — 채택 판정과 제출에는 사용하지 않음

In [7]:
fm = (tr.split == 'fit').values; vm = ~fm
trf = tr[fm].reset_index(drop=True); yf = y[fm]
pv = LR().fit(trf[FEATS], yf).predict_proba(tr[vm][FEATS])[:, 1]
tv = pick_t(yf, loso_proba(trf, yf, list(LeaveOneGroupOut().split(trf, yf, g[fm]))))
yv = y[vm]; stv = st[vm]
for name, p in [('A', predA[vm]), ('B', (pv > 0.5).astype(int)), (f'C (t={tv})', (pv > tv).astype(int))]:
    print(f'{name:10s} valid macroF1 {mf1(yv, p):.4f}  moving {mf1(yv[~stv], p[~stv]):.4f}  stationary {mf1(yv[stv], p[stv]):.4f}')
print(f'total elapsed {time.time()-T0:.1f}s')

A          valid macroF1 0.8675  moving 0.8982  stationary 0.6700
B          valid macroF1 0.8403  moving 0.9107  stationary 0.6459
C (t=0.55) valid macroF1 0.8723  moving 0.9107  stationary 0.7273
total elapsed 1.6s
